In [ ]:
# ==============================================================================
# 0. SETUP & PIPELINE DEPENDENCIES
# ==============================================================================
# --- Fix: Pin Pillow < 12.0 to prevent Torchvision typing ImportError ---
!pip install --upgrade --force-reinstall --no-deps "Pillow>=10.4.0,<12.0"
!pip install -q huggingface_hub timm einops opencv-python-headless lpips torchmetrics

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import glob, shutil, gc, re, json, random, math
from functools import partial
from typing import Literal, Tuple, Optional

import cv2
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint as checkpoint
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from torchvision import transforms
from torch.nn import LayerNorm
from torch.nn.init import trunc_normal_

from huggingface_hub import snapshot_download
from timm.models.layers import DropPath
import lpips as lpips_lib
from torchmetrics.image import PeakSignalNoiseRatio, StructuralSimilarityIndexMeasure

# Performance flags
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# ==============================================================================
# 1. DATASET DOWNLOAD & PREPROCESSING (MiDaS Depth + Masks)
# ==============================================================================
print("\n--- 1. Downloading RealBokeh_3MP subset (~100 train, ~20 test) ---")
RAW_DIR = "raw_realbokeh"
train_patterns = [f"train/gt/{i}/*" for i in range(1, 21)]
test_patterns  = [f"test/gt/{i}/*" for i in range(1, 5)]
snapshot_download(
    repo_id="timseizinger/RealBokeh_3MP",
    repo_type="dataset",
    allow_patterns=train_patterns + test_patterns,
    local_dir=RAW_DIR,
    max_workers=8,
)

print(f"--- 2. Loading MiDaS Small into VRAM... (device={device}) ---")
midas = torch.hub.load("intel-isl/MiDaS", "MiDaS_small").to(device).eval()
transform = torch.hub.load("intel-isl/MiDaS", "transforms").small_transform

def get_fstop(fname):
    m = re.search(r'_f([0-9.]+)\.JPG', fname, re.IGNORECASE)
    return float(m.group(1)) if m else 0.0

for split in ["train", "test"]:
    OUT_DIR = f"dataset/RealBokeh_3MP/{split}"
    for sub in ["rgb", "bokeh", "depth", "mask"]:
        os.makedirs(os.path.join(OUT_DIR, sub), exist_ok=True)

    groups = glob.glob(f"{RAW_DIR}/{split}/gt/*/")
    for group in tqdm(groups, desc=f"Processing {split}"):
        group_id = os.path.basename(os.path.normpath(group))
        images = glob.glob(os.path.join(group, "*.JPG"))
        if not images:
            continue

        images.sort(key=get_fstop, reverse=True)
        sharp_path, bokeh_paths = images[0], images[1:]

        sharp_img = cv2.imread(sharp_path)
        sharp_rgb = cv2.cvtColor(sharp_img, cv2.COLOR_BGR2RGB)
        cv2.imwrite(os.path.join(OUT_DIR, "rgb", f"{group_id}.png"), sharp_img)

        for bp in bokeh_paths:
            fstop = get_fstop(bp)
            shutil.copy(bp, os.path.join(OUT_DIR, "bokeh", f"{group_id}_f{fstop}.png"))

        input_batch = transform(sharp_rgb).to(device)
        with torch.no_grad():
            pred = midas(input_batch)
            pred = torch.nn.functional.interpolate(
                pred.unsqueeze(1), size=sharp_rgb.shape[:2], mode="bicubic", align_corners=False
            ).squeeze()

        depth_map = cv2.normalize(pred.cpu().numpy(), None, 0, 255, norm_type=cv2.NORM_MINMAX, dtype=cv2.CV_8U)
        cv2.imwrite(os.path.join(OUT_DIR, "depth", f"{group_id}.png"), depth_map)

        _, mask = cv2.threshold(depth_map, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=3)
        cv2.imwrite(os.path.join(OUT_DIR, "mask", f"{group_id}.png"), mask)

print("Cleaning up VRAM...")
gc.collect()
torch.cuda.empty_cache()
print("Dataset ready at dataset/RealBokeh_3MP!\n")


# ==============================================================================
# 2. NEURAL NETWORK UTILITIES & BLOCKS
# ==============================================================================
class IdentityMod(nn.Module):
    def forward(self, x=None, *args, **kwargs) -> torch.Tensor:
        return x

class ConcatTensors(nn.Module):
    def forward(self, x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        return torch.cat((x, y), dim=1)

class SkipConnection(nn.Module):
    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        return x + skip

class ApplyVectorWeights(nn.Module):
    def forward(self, x: torch.Tensor, weights: torch.Tensor) -> torch.Tensor:
        return x * weights

class ChannelEmbeddingCompression(nn.Module):
    def __init__(self, embed_dim, embed_dim_next):
        super().__init__()
        self.patch_unembed = PatchUnEmbedIR(embed_dim=embed_dim)
        self.conv = nn.Conv2d(embed_dim, embed_dim_next, 1, 1, 0)
        self.patch_embed = PatchEmbedIR(embed_dim=embed_dim_next)

    def forward(self, x, x_size):
        x = self.patch_unembed(x, x_size)
        x = self.conv(x)
        x = self.patch_embed(x)
        return x

class InvertedConvolution(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, padding='same', bias=True):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=in_channels, out_channels=out_channels,
                               kernel_size=1, padding=0, stride=1, groups=1, bias=bias)
        self.conv2 = nn.Conv2d(in_channels=out_channels, out_channels=out_channels, kernel_size=kernel_size,
                               padding=padding, stride=1, groups=out_channels, bias=bias)
    def forward(self, x):
        return self.conv2(self.conv1(x))

class DWConv2d(nn.Module):
    def __init__(self, dim, kernel_size, stride, padding):
        super().__init__()
        self.conv = nn.Conv2d(dim, dim, kernel_size, stride, padding, groups=dim)

    @torch.amp.autocast('cuda', enabled=False)
    def forward(self, x: torch.Tensor):
        x = x.float()
        x = x.permute(0, 3, 1, 2)
        x = self.conv(x)
        return x.permute(0, 2, 3, 1)

class ChannelAttention(nn.Module):
    def __init__(self, num_channel: int):
        super().__init__()
        self.model = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels=num_channel, out_channels=num_channel // 2, kernel_size=1, padding=0, stride=1, bias=True),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=num_channel // 2, out_channels=num_channel, kernel_size=1, padding=0, stride=1, bias=True),
            nn.Sigmoid()
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)

class SimplifiedChannelAttention(nn.Module):
    def __init__(self, num_channel, apply_att_weights=False):
        super().__init__()
        self.model = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels=num_channel, out_channels=num_channel, kernel_size=1, padding=0, stride=1, bias=True),
        )
        self.apply1 = ApplyVectorWeights() if apply_att_weights else IdentityMod()
    def forward(self, x: torch.Tensor, att_weights: torch.Tensor = None) -> torch.Tensor:
        return self.apply1(x=self.model(x), weights=att_weights)

class ApertureAwareAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, value_factor=1):
        super().__init__()
        self.factor = value_factor
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = self.embed_dim * self.factor // num_heads
        self.key_dim = self.embed_dim // num_heads
        self.scaling = self.key_dim ** -0.5

        self.q_proj = nn.Linear(embed_dim, embed_dim, bias=True)
        self.k_proj = nn.Linear(embed_dim, embed_dim, bias=True)
        self.v_proj = nn.Linear(embed_dim, embed_dim * self.factor, bias=True)
        self.lepe = DWConv2d(embed_dim, 5, 1, 2)
        self.out_proj = nn.Linear(embed_dim * self.factor, embed_dim, bias=True)
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_normal_(self.q_proj.weight, gain=2 ** -2.5)
        nn.init.xavier_normal_(self.k_proj.weight, gain=2 ** -2.5)
        nn.init.xavier_normal_(self.v_proj.weight, gain=2 ** -2.5)
        nn.init.xavier_normal_(self.out_proj.weight)
        nn.init.constant_(self.out_proj.bias, 0.0)

    @torch.amp.autocast('cuda', enabled=False)
    def forward(self, x: torch.Tensor, rel_pos):
        x = x.float()
        bsz, h, w, _ = x.size()
        mask_h, mask_w = rel_pos
        mask_h = mask_h.float()
        mask_w = mask_w.float()

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)
        lepe = self.lepe(v)

        k *= self.scaling
        qr = q.view(bsz, h, w, self.num_heads, self.key_dim).permute(0, 3, 1, 2, 4)
        kr = k.view(bsz, h, w, self.num_heads, self.key_dim).permute(0, 3, 1, 2, 4)

        qr_w = qr.transpose(1, 2)
        kr_w = kr.transpose(1, 2)
        v = v.reshape(bsz, h, w, self.num_heads, -1).permute(0, 1, 3, 2, 4)

        v_out_w = []
        chunk_size_h = 32
        mask_w_expand = mask_w.unsqueeze(1)
        for i in range(0, h, chunk_size_h):
            qk_mat_w = qr_w[:, i:i+chunk_size_h] @ kr_w[:, i:i+chunk_size_h].transpose(-1, -2)
            qk_mat_w = qk_mat_w + mask_w_expand
            qk_mat_w = torch.softmax(qk_mat_w, -1)
            v_chunk = qk_mat_w @ v[:, i:i+chunk_size_h]
            v_out_w.append(v_chunk)
        v = torch.cat(v_out_w, dim=1)

        qr_h = qr.permute(0, 3, 1, 2, 4)
        kr_h = kr.permute(0, 3, 1, 2, 4)
        v = v.permute(0, 3, 2, 1, 4)

        v_out_h = []
        chunk_size_w = 32
        mask_h_expand = mask_h.unsqueeze(1)
        for i in range(0, w, chunk_size_w):
            qk_mat_h = qr_h[:, i:i+chunk_size_w] @ kr_h[:, i:i+chunk_size_w].transpose(-1, -2)
            qk_mat_h = qk_mat_h + mask_h_expand
            qk_mat_h = torch.softmax(qk_mat_h, -1)
            v_chunk = qk_mat_h @ v[:, i:i+chunk_size_w]
            v_out_h.append(v_chunk)

        output = torch.cat(v_out_h, dim=1)
        output = output.permute(0, 3, 1, 2, 4).flatten(-2, -1)
        output = output + lepe
        output = self.out_proj(output)
        return output

class SimpleGate(nn.Module):
    def forward(self, x):
        x1, x2 = x.chunk(2, dim=1)
        return x1 * x2

class LayerNorm2d(nn.GroupNorm):
    def __init__(self, channels, eps=1e-4):
        super().__init__(1, channels, eps=eps)

class DynRelPos2d(nn.Module):
    def __init__(self, embed_dim, num_heads, initial_value, heads_range):
        super().__init__()
        angle = 1.0 / (10000 ** torch.linspace(0, 1, embed_dim // num_heads // 2))
        angle = angle.unsqueeze(-1).repeat(1, 2).flatten()
        self.initial_value = initial_value
        self.heads_range = heads_range
        self.num_heads = num_heads
        self.register_buffer('angle', angle)

    @torch.amp.autocast('cuda', enabled=False)
    def generate_1d_decay(self, l: int, range_factor: torch.Tensor):
        range_factor = abs(range_factor).float()
        bs = range_factor.size(0)
        heads_ranges = self.heads_range * torch.arange(self.num_heads, dtype=torch.float) / self.num_heads
        heads_ranges = heads_ranges.to(range_factor.device)
        range_factor = torch.sqrt(torch.sqrt(range_factor))[:, None]
        ranges = (-self.initial_value - heads_ranges.repeat(bs, 1) * range_factor)
        decay = torch.log(1 - 2 ** ranges)
        index = torch.arange(l).to(decay)
        mask = (index[:, None] - index[None, :]).abs()[None, None, :, :]
        return mask * decay[:, :, None, None]

    def forward(self, slen: Tuple[int], range_factor: torch.Tensor):
        mask_h = self.generate_1d_decay(slen[0], range_factor=range_factor)
        mask_w = self.generate_1d_decay(slen[1], range_factor=range_factor)
        return mask_h, mask_w

class PatchEmbedIR(nn.Module):
    def __init__(self, embed_dim=96, norm_layer=None):
        super().__init__()
        self.norm = nn.LayerNorm(embed_dim) if norm_layer is not None else None
    def forward(self, x):
        x = x.permute(0, 2, 3, 1)
        if self.norm is not None:
            x = self.norm(x)
        return x

class PatchUnEmbedIR(nn.Module):
    def __init__(self, embed_dim=96):
        super().__init__()
        self.embed_dim = embed_dim
    def forward(self, x, x_size):
        B, H, W, C = x.shape
        x = x.permute(0, 3, 1, 2)
        return x.view(B, self.embed_dim, x_size[0], x_size[1])

class ApertureEncoder(nn.Module):
    def __init__(self, d_embed=64, num_freqs=8):
        super().__init__()
        self.num_freqs = num_freqs
        in_dim = 2 * num_freqs
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, d_embed // 2),
            nn.GELU(),
            nn.Linear(d_embed // 2, d_embed)
        )
        freq_bands = 2 ** torch.linspace(0, num_freqs - 1, num_freqs)
        self.register_buffer("freq_bands", freq_bands)
    def forward(self, x):
        x_freq = x * self.freq_bands[None, :] * math.pi
        fourier_features = torch.cat([torch.sin(x_freq), torch.cos(x_freq)], dim=-1)
        return self.mlp(fourier_features)

class FiLMLayer(nn.Module):
    def __init__(self, in_channels, d_embed=64):
        super().__init__()
        self.fc = nn.Linear(d_embed, in_channels * 2)
        nn.init.zeros_(self.fc.weight)
        nn.init.zeros_(self.fc.bias)
    def forward(self, x, condition):
        scale_shift = self.fc(condition)
        scale, shift = scale_shift.chunk(2, dim=1)
        if x.dim() == 4:
            if x.shape[1] == scale.shape[1]:
                scale = scale.view(-1, scale.shape[1], 1, 1)
                shift = shift.view(-1, shift.shape[1], 1, 1)
            else:
                scale = scale.view(-1, 1, 1, scale.shape[1])
                shift = shift.view(-1, 1, 1, shift.shape[1])
        return x * (1 + scale) + shift

class FocalPriorGenerator(nn.Module):
    def forward(self, depth, mask):
        focal_dist = (depth * mask).sum(dim=(2, 3), keepdim=True) / (mask.sum(dim=(2, 3), keepdim=True) + 1e-4)
        focal_map = torch.abs(depth - focal_dist)
        soft_mask = torch.exp(-focal_map)
        return focal_map, soft_mask

class FusionStem(nn.Module):
    def __init__(self, in_channels=7, d_embed=64):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Conv2d(in_channels, d_embed, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(d_embed, d_embed, kernel_size=3, padding=1),
        )
    def forward(self, rgb, depth, mask, focal_map, soft_mask=None):
        if soft_mask is not None:
            x = torch.cat([rgb, depth, mask, focal_map, soft_mask], dim=1)
        else:
            x = torch.cat([rgb, depth, mask, focal_map], dim=1)
        return self.proj(x)

def get_activation(activation_type='Identity'):
    if activation_type == 'GELU': return nn.GELU()
    elif activation_type == 'SG': return SimpleGate()
    elif activation_type == 'Identity': return nn.Identity()
    elif activation_type == 'ReLU': return nn.ReLU()
    elif activation_type == 'LReLU': return nn.LeakyReLU()
    elif activation_type == 'Tanh': return nn.Tanh()
    elif activation_type == 'Sigmoid': return nn.Sigmoid()
    else: raise NotImplementedError(f'Activation type {activation_type} is not implemented.')

def get_cnn_attention(attention_type=None):
    if attention_type == 'CA': return ChannelAttention
    elif attention_type == 'SCA': return SimplifiedChannelAttention
    else: raise NotImplementedError(f'Attention type {attention_type} is not implemented.')

class FeedForwardNetwork(nn.Module):
    def __init__(self, embed_dim, ffn_dim, activation_fn=F.gelu, dropout=0.0, activation_dropout=0.0, layernorm_eps=1e-4, subln=False, subconv=False):
        super().__init__()
        self.embed_dim = embed_dim
        self.activation_fn = activation_fn
        self.activation_dropout_module = torch.nn.Dropout(activation_dropout)
        self.dropout_module = torch.nn.Dropout(dropout)
        self.fc1 = nn.Linear(self.embed_dim, ffn_dim)
        self.fc2 = nn.Linear(ffn_dim, self.embed_dim)
        self.ffn_layernorm = nn.LayerNorm(ffn_dim, eps=layernorm_eps) if subln else None
        self.dwconv = DWConv2d(ffn_dim, 3, 1, 1) if subconv else None

    def forward(self, x: torch.Tensor):
        x = self.fc1(x)
        x = self.activation_fn(x)
        x = self.activation_dropout_module(x)
        if self.dwconv is not None:
            residual = x
            x = self.dwconv(x) + residual
        if self.ffn_layernorm is not None:
            x = self.ffn_layernorm(x)
        return self.dropout_module(self.fc2(x))

class ApertureAttentionBlock(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int, ffn_dim: int, drop_path=0., layerscale=False, norm_layer=nn.LayerNorm, layer_init_values=1e-5):
        super().__init__()
        self.layerscale = layerscale
        self.embed_dim = embed_dim
        self.attention_layer_norm = norm_layer(self.embed_dim, eps=1e-4)
        self.attention = ApertureAwareAttention(embed_dim, num_heads)
        self.drop_path = DropPath(drop_path)
        self.final_layer_norm = norm_layer(self.embed_dim, eps=1e-4)
        self.ffn = FeedForwardNetwork(embed_dim, ffn_dim)
        self.pos = DWConv2d(embed_dim, 3, 1, 1)
        if layerscale:
            self.gamma_1 = nn.Parameter(layer_init_values * torch.ones(1, 1, 1, embed_dim), requires_grad=True)
            self.gamma_2 = nn.Parameter(layer_init_values * torch.ones(1, 1, 1, embed_dim), requires_grad=True)

    def forward(self, x: torch.Tensor, attention_rel_pos=None):
        x = x + self.pos(x)
        if self.layerscale:
            x = x + self.drop_path(self.gamma_1 * self.attention(self.attention_layer_norm(x), attention_rel_pos))
            x = x + self.drop_path(self.gamma_2 * self.ffn(self.final_layer_norm(x)))
        else:
            x = x + self.drop_path(self.attention(self.attention_layer_norm(x), attention_rel_pos))
            x = x + self.drop_path(self.ffn(self.final_layer_norm(x)))
        return x

class BasicLayer(nn.Module):
    def __init__(self, embed_dim, depth, num_heads, init_value: float, heads_range: float, ffn_dim=96, drop_path=0., norm_layer=nn.LayerNorm, use_checkpoint=False, layerscale=False, layer_init_values=1e-5):
        super().__init__()
        self.embed_dim = embed_dim
        self.depth = depth
        self.use_checkpoint = use_checkpoint
        self.Relpos = DynRelPos2d(embed_dim, num_heads, init_value, heads_range)
        self.blocks = nn.ModuleList([
            ApertureAttentionBlock(embed_dim=embed_dim, num_heads=num_heads, ffn_dim=ffn_dim,
                                   layerscale=layerscale, norm_layer=norm_layer, layer_init_values=layer_init_values,
                                   drop_path=drop_path[i] if isinstance(drop_path, list) else drop_path)
            for i in range(depth)])

    def forward(self, x, att_range_factor=None):
        b, h, w, d = x.size()
        rel_pos = self.Relpos((h, w), range_factor=att_range_factor)
        for blk in self.blocks:
            if self.use_checkpoint:
                tmp_blk = partial(blk, attention_rel_pos=rel_pos)
                x = checkpoint.checkpoint(tmp_blk, x, use_reentrant=False)
            else:
                x = blk(x, attention_rel_pos=rel_pos)
        return x

class ResidualBlock(nn.Module):
    def __init__(self, embed_dim, embed_dim_next, depth, num_heads, heads_range, init_value, ffn_dim=96, layerscale=False, layer_init_values=1e-5, drop_path=0., norm_layer=nn.LayerNorm, use_checkpoint=False, resi_connection='1conv', use_pos_map=False):
        super().__init__()
        self.embed_dim = embed_dim
        self.residual_group = BasicLayer(embed_dim=embed_dim, depth=depth, num_heads=num_heads, heads_range=heads_range, init_value=init_value, drop_path=drop_path, ffn_dim=ffn_dim, norm_layer=norm_layer, use_checkpoint=use_checkpoint, layerscale=layerscale, layer_init_values=layer_init_values)
        embed_dim_conv_in = embed_dim + 2 if use_pos_map else embed_dim
        if resi_connection == '1conv':
            self.conv = nn.Conv2d(embed_dim_conv_in, embed_dim, 3, 1, 1)
        elif resi_connection == '3conv':
            self.conv = nn.Sequential(nn.Conv2d(embed_dim_conv_in, embed_dim // 4, 3, 1, 1),
                                      nn.LeakyReLU(negative_slope=0.2, inplace=True),
                                      nn.Conv2d(embed_dim // 4, embed_dim // 4, 1, 1, 0),
                                      nn.LeakyReLU(negative_slope=0.2, inplace=True),
                                      nn.Conv2d(embed_dim // 4, embed_dim, 3, 1, 1))
        self.patch_embed = PatchEmbedIR(embed_dim=embed_dim)
        self.patch_unembed = PatchUnEmbedIR(embed_dim=embed_dim)
        self.final_op = ChannelEmbeddingCompression(embed_dim, embed_dim_next) if (embed_dim != embed_dim_next) else IdentityMod()
        self.use_pos_map = use_pos_map

    def forward(self, x, x_size, pos_map=None, att_range_factor=None, **kwargs):
        rb_stl_out = self.residual_group(x, att_range_factor)
        rb_unembed = self.patch_unembed(rb_stl_out, x_size)
        rb_unembed = torch.cat((rb_unembed, pos_map), dim=1) if self.use_pos_map and pos_map is not None else rb_unembed
        rb_last_conv = self.conv(rb_unembed)
        rb_re_enbed = self.patch_embed(rb_last_conv)
        rb_res = rb_re_enbed + x
        return self.final_op(rb_res, x_size)

class BlockMod(nn.Module):
    def __init__(self, channels: int, dw_expand: float = 1., ffn_expand: int = 2, drop_out_rate: float = 0.,
                 attention_type: Literal['CA', 'SCA'] = 'CA',
                 activation_type: Literal['GELU', 'SG', 'Identity', 'ReLU', 'LReLU', 'Tanh', 'Sigmoid'] = 'GELU',
                 inverted_conv: bool = True, kernel_size: int = 3,
                 use_pos_map: bool = False, depth: int = None, d_embed: int = 64):
        super().__init__()
        dw_channel = int(channels * dw_expand) if int(channels * dw_expand) % 2 == 0 else int(channels * dw_expand) + 1
        self.cat = ConcatTensors() if use_pos_map else IdentityMod()
        if inverted_conv:
            self.conv1 = InvertedConvolution(in_channels=channels + 2 if use_pos_map else channels, out_channels=dw_channel, kernel_size=kernel_size, padding='same', bias=True)
        else:
            self.conv1 = nn.Conv2d(in_channels=channels + 2 if use_pos_map else channels, out_channels=dw_channel, kernel_size=kernel_size, padding='same', stride=1, bias=True)
        self.film1 = FiLMLayer(dw_channel, d_embed)
        self.activation = get_activation(activation_type)
        self.attention = get_cnn_attention(attention_type)(dw_channel // 2 if activation_type == 'SG' else dw_channel)
        self.conv2 = nn.Conv2d(in_channels=dw_channel // 2 if activation_type == 'SG' else dw_channel, out_channels=channels, kernel_size=1, padding=0, stride=1, groups=1, bias=True)
        ffn_channel = math.floor(ffn_expand * channels)
        self.conv3 = nn.Conv2d(in_channels=channels, out_channels=ffn_channel, kernel_size=1, padding=0, stride=1, groups=1, bias=True)
        self.conv4 = nn.Conv2d(in_channels=ffn_channel // 2 if activation_type == 'SG' else ffn_channel, out_channels=channels, kernel_size=1, padding=0, stride=1, groups=1, bias=True)
        self.norm1 = LayerNorm2d(channels)
        self.norm2 = LayerNorm2d(channels)
        self.dropout1 = nn.Dropout(drop_out_rate) if drop_out_rate > 0. else nn.Identity()
        self.dropout2 = nn.Dropout(drop_out_rate) if drop_out_rate > 0. else nn.Identity()
        self.beta = nn.Parameter(torch.zeros((1, channels, 1, 1)), requires_grad=True)
        self.gamma = nn.Parameter(torch.zeros((1, channels, 1, 1)), requires_grad=True)
        self.use_pos_map = use_pos_map
        self.depth = depth

    def forward(self, source: torch.Tensor, pos_map: torch.Tensor = None, e_f: torch.Tensor = None) -> torch.Tensor:
        x = self.norm1(source)
        x = self.cat(x, pos_map)
        x = self.conv1(x)
        if e_f is not None:
            x = self.film1(x, e_f)
        x = self.activation(x)
        x = x * self.attention(x)
        x = self.dropout1(self.conv2(x))
        y = source + x * self.beta
        x = self.conv3(self.norm2(y))
        x = self.activation(x)
        x = self.dropout2(self.conv4(x))
        return y + x * self.gamma


# ==============================================================================
# 3. BOKEHLICIOUS BACKBONE & COMPLETE STABILIZED HAFT MODEL
# ==============================================================================
class Bokehlicious(nn.Module):
    def __init__(self,
                 in_chans=3, img_range=1., dynamic_conv_k=4,
                 in_stage_use_pos_map: bool = True,
                 in_stage_use_bokeh_strength_map: bool = False,
                 u_width=32, u_depth=2, u_block_config=None, u_skip_connections=None,
                 enc_blk_nums=None, enc_blks_use_pos_map=None,
                 embed_dims=None, depths=None, num_heads=None,
                 init_values=None, heads_ranges=None, mlp_ratios=None,
                 drop_path_rate=0.1, norm_layer=LayerNorm, patch_norm=True,
                 use_checkpoints=None, chunkwise_recurrents=None, layerscales=None,
                 layer_init_values=1e-6, positional_dfe=False, use_dfe_norm_layer=True,
                 positional_conv_last=False, dec_blk_nums=None, dec_blks_use_pos_map=None,
                 out_stage_use_pos_map: bool = True, **kwargs):
        super().__init__()
        self.in_chans = in_chans
        self.img_range = img_range
        self.in_stage_use_pos_map = in_stage_use_pos_map
        self.in_stage_use_bokeh_strength_map = in_stage_use_bokeh_strength_map
        self.u_width = u_width
        self.u_depth = u_depth
        self.u_block_config = u_block_config or {'dw_expand': 1., 'ffn_expand': 2., 'drop_out_rate': 0, 'attention_type': 'CA', 'activation_type': 'GELU', 'kernel_size': 3, 'inverted_conv': True}
        self.u_skip_connections = u_skip_connections or [True for _ in range(u_depth)]
        self.enc_blk_nums = enc_blk_nums or [1 for _ in range(u_depth)]
        self.enc_blks_use_pos_map = enc_blks_use_pos_map or [True for _ in range(u_depth)]

        self.num_blocks = len(embed_dims)
        self.embed_dims = embed_dims or [192 for _ in range(self.num_blocks)]
        self.depths = depths or [6 for _ in range(self.num_blocks)]
        self.num_heads = num_heads or [6 for _ in range(self.num_blocks)]
        self.init_values = init_values or [2 for _ in range(self.num_blocks)]
        self.heads_ranges = heads_ranges or [6 for _ in range(self.num_blocks)]
        self.mlp_ratios = mlp_ratios or [2 for _ in range(self.num_blocks)]
        self.drop_path_rate = drop_path_rate
        self.chunkwise_recurrents = chunkwise_recurrents or [True for _ in range(self.num_blocks)]
        self.layerscales = layerscales or [False for _ in range(self.num_blocks)]
        self.use_checkpoints = use_checkpoints or [False for _ in range(self.num_blocks)]
        self.positional_dfe = positional_dfe
        self.layer_init_values = layer_init_values
        self.patch_norm = patch_norm
        self.norm_layer = norm_layer
        self.use_dfe_norm_layer = use_dfe_norm_layer
        self.positional_conv_last = positional_conv_last
        self.dec_blk_nums = dec_blk_nums or [1 for _ in range(u_depth)]
        self.dec_blks_use_pos_map = dec_blks_use_pos_map or [True for _ in range(u_depth)]
        self.out_stage_use_pos_map = out_stage_use_pos_map

        extra_in_channels = (2 if self.in_stage_use_pos_map else 0) + (1 if self.in_stage_use_bokeh_strength_map else 0)
        self.use_coc_map = kwargs.get('use_coc_map', True)
        extra_in_channels += 1 if self.use_coc_map else 0

        self.in_stage = nn.Conv2d(in_channels=self.in_chans + extra_in_channels, out_channels=self.u_width, kernel_size=3, padding=1, stride=1, groups=1, bias=True)
        self.in_stage_2 = nn.Conv2d(in_channels=self.u_width, out_channels=self.u_width, kernel_size=3, padding=1, stride=1, groups=1, bias=True)

        self.encoders = nn.ModuleList()
        self.downs = nn.ModuleList()
        chan = self.u_width
        u_depths = range(0, self.u_depth)
        for num, depth, use_pos_map in zip(self.enc_blk_nums, u_depths, self.enc_blks_use_pos_map):
            self.encoders.append(nn.ModuleList([BlockMod(chan, **self.u_block_config, depth=depth, use_pos_map=use_pos_map) for _ in range(num)]))
            self.downs.append(nn.Conv2d(chan, chan * 2, 2, 2))
            chan = chan * 2

        self.conv_prep = nn.Conv2d(chan, self.embed_dims[0], 3, 1, 1)
        if in_chans == 3:
            self.mean = torch.Tensor((0.4488, 0.4371, 0.4040)).view(1, 3, 1, 1)
        else:
            self.mean = torch.zeros(1, 1, 1, 1)

        self.patch_embed = PatchEmbedIR(embed_dim=self.embed_dims[0], norm_layer=self.norm_layer if self.patch_norm else None)
        self.patch_unembed = PatchUnEmbedIR(embed_dim=self.embed_dims[-1])

        dpr = [x.item() for x in torch.linspace(0, self.drop_path_rate, sum(self.depths))]
        self.blocks = nn.ModuleList()
        for i_block in range(self.num_blocks):
            self.blocks.append(ResidualBlock(
                embed_dim=self.embed_dims[i_block],
                embed_dim_next=self.embed_dims[i_block + 1] if i_block < self.num_blocks - 1 else self.embed_dims[i_block],
                depth=self.depths[i_block], num_heads=self.num_heads[i_block], init_value=self.init_values[i_block],
                heads_range=self.heads_ranges[i_block], ffn_dim=int(self.mlp_ratios[i_block] * self.embed_dims[i_block]),
                drop_path=dpr[sum(self.depths[:i_block]):sum(self.depths[:i_block + 1])], norm_layer=self.norm_layer,
                use_checkpoint=self.use_checkpoints[i_block], layerscale=self.layerscales[i_block],
                layer_init_values=self.layer_init_values, use_pos_map=self.positional_dfe,
            ))
        self.norm = nn.LayerNorm(self.embed_dims[-1], eps=1e-4) if self.use_dfe_norm_layer else IdentityMod()
        self.conv_after_body = nn.Conv2d(self.embed_dims[-1], self.embed_dims[0], 3, 1, 1)

        conv_last_extra_channels = 2 if self.positional_conv_last else 0
        self.conv_last = nn.Conv2d(self.embed_dims[0] + conv_last_extra_channels, chan, 3, 1, 1)

        self.decoders = nn.ModuleList()
        self.skips = nn.ModuleList()
        self.ups = nn.ModuleList()
        for num, use_pos_map, skip_connection, depth in zip(self.dec_blk_nums, self.dec_blks_use_pos_map, self.u_skip_connections, u_depths.__reversed__()):
            self.ups.append(nn.Sequential(nn.Conv2d(in_channels=chan, out_channels=2 * chan, kernel_size=1, bias=False), nn.PixelShuffle(2)))
            self.skips.append(SkipConnection() if skip_connection else IdentityMod())
            chan = chan // 2
            self.decoders.append(nn.ModuleList([BlockMod(chan, **self.u_block_config, use_pos_map=use_pos_map, depth=depth) for x in range(num)]))

        extra_out_channels = 2 if self.out_stage_use_pos_map else 0
        self.out_stage = nn.Conv2d(in_channels=u_width + extra_out_channels, out_channels=in_chans, kernel_size=3, padding=1, stride=1, groups=1, bias=True)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Conv2d):
            if m is self.in_stage and self.use_coc_map:
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
                with torch.no_grad():
                    m.weight[:, -1, :, :] = 0.0
        elif isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            try:
                nn.init.constant_(m.bias, 0)
                nn.init.constant_(m.weight, 1.0)
            except:
                pass

    def forward_features(self, x, pos_map=None, bokeh_strength=None):
        x_size = (x.shape[2], x.shape[3])
        x = self.patch_embed(x)
        for block in self.blocks:
            x = block(x, x_size, pos_map=pos_map, att_range_factor=bokeh_strength)
        x = self.norm(x)
        return self.patch_unembed(x, x_size)

    def forward(self, source: torch.Tensor, bokeh_strength: torch.Tensor = None, pos_map: torch.Tensor = None,
                bokeh_strength_map: torch.Tensor = None, coc_map: torch.Tensor = None, **kwargs):
        self.mean = self.mean.type_as(source)
        source_mean = (source - self.mean) * self.img_range
        x = torch.cat((source_mean, pos_map), dim=1) if self.in_stage_use_pos_map and pos_map is not None else source_mean
        x = torch.cat((x, bokeh_strength_map), dim=1) if self.in_stage_use_bokeh_strength_map and bokeh_strength_map is not None else x
        if self.use_coc_map and coc_map is not None:
            x = torch.cat((x, coc_map), dim=1)

        x = self.in_stage(x)
        x = self.in_stage_2(x)

        encs = []
        for encoder, down, use_pos_map_e, depth in zip(self.encoders, self.downs, self.enc_blks_use_pos_map, range(0, self.u_depth)):
            pos_map_e = F.interpolate(pos_map, scale_factor=1 / 2 ** depth, mode='bilinear', align_corners=False) if (use_pos_map_e and pos_map is not None) else None
            for blk in encoder:
                x = blk(x, pos_map=pos_map_e)
            encs.append(x)
            x = down(x)

        x_prep = self.conv_prep(x)
        pos_map_t = F.interpolate(pos_map, scale_factor=1 / 2 ** self.u_depth, mode='bilinear', align_corners=False) if (self.positional_dfe or self.positional_conv_last) and pos_map is not None else None
        x_after_body = self.forward_features(x_prep, bokeh_strength=bokeh_strength, pos_map=pos_map_t)
        res = self.conv_after_body(x_after_body) + x_prep

        res = torch.cat((res, pos_map_t), dim=1) if self.positional_conv_last and pos_map_t is not None else res
        x = x + self.conv_last(res)

        for decoder, up, skip, enc_skip, use_pos_map_d, depth in zip(self.decoders, self.ups, self.skips, encs[::-1], self.dec_blks_use_pos_map, range(0, self.u_depth).__reversed__()):
            pos_map_d = F.interpolate(pos_map, scale_factor=1 / 2 ** depth, mode='bilinear', align_corners=False) if (use_pos_map_d and pos_map is not None) else None
            x = up(x)
            x = skip(x, enc_skip)
            for blk in decoder:
                x = blk(x, pos_map=pos_map_d)

        x = torch.cat((x, pos_map), dim=1) if self.out_stage_use_pos_map and pos_map is not None else x
        x = self.out_stage(x)
        x = x / self.img_range + self.mean
        return x + source


class HAFT_Bokehlicious(Bokehlicious):
    def __init__(self, d_embed=64, use_refinement=True, use_coc_map=True, in_stage_use_pos_map=True, **kwargs):
        kwargs['in_stage_use_pos_map'] = in_stage_use_pos_map
        kwargs['use_coc_map'] = use_coc_map
        super().__init__(**kwargs)
        self.d_embed = d_embed
        self.use_refinement = use_refinement
        self.use_coc_map = use_coc_map
        self.in_stage_use_pos_map = in_stage_use_pos_map
        self.aperture_encoder = ApertureEncoder(d_embed=self.d_embed)

        if self.use_refinement:
            self.prior_gen = FocalPriorGenerator()
            self.fusion_stem = FusionStem(in_channels=7, d_embed=64)
            self.refinement_blocks = ResidualBlock(
                embed_dim=64, embed_dim_next=64, depth=2, num_heads=2,
                init_value=2, heads_range=6, ffn_dim=128, norm_layer=LayerNorm, 
                # FIX: Tie this to the dynamic flag instead of hardcoding True
                use_pos_map=self.in_stage_use_pos_map
            )
            # ZERO-INIT & SCALED RESIDUAL: Guaranteed 0 starting contribution
            self.refinement_out = nn.Conv2d(64, self.in_chans, kernel_size=3, padding=1)
            nn.init.zeros_(self.refinement_out.weight)
            nn.init.zeros_(self.refinement_out.bias)
            self.res_scale = nn.Parameter(torch.zeros(1))

    def forward(self, source, bokeh_strength=None, pos_map=None, bokeh_strength_map=None, depth=None, mask=None, **kwargs):
        if not getattr(self, 'in_stage_use_pos_map', True):
            pos_map = None

        f_val = bokeh_strength.view(-1, 1) if bokeh_strength is not None else torch.ones(source.shape[0], 1).to(source.device)
        e_f = self.aperture_encoder(f_val)

        self.mean = self.mean.type_as(source)
        source_mean = (source - self.mean) * self.img_range

        x = torch.cat((source_mean, pos_map), dim=1) if self.in_stage_use_pos_map and pos_map is not None else source_mean
        x = torch.cat((x, bokeh_strength_map), dim=1) if self.in_stage_use_bokeh_strength_map and bokeh_strength_map is not None else x
        if getattr(self, 'use_coc_map', True) and kwargs.get('coc_map') is not None:
            x = torch.cat((x, kwargs['coc_map']), dim=1)

        x = self.in_stage(x)
        x = self.in_stage_2(x)

        encs = []
        for encoder, down, use_pos_map_e, depth_lvl in zip(self.encoders, self.downs, self.enc_blks_use_pos_map, range(0, self.u_depth)):
            pos_map_e = F.interpolate(pos_map, scale_factor=1 / 2 ** depth_lvl, mode='bilinear', align_corners=False) if (use_pos_map_e and pos_map is not None) else None
            for blk in encoder:
                x = blk(x, pos_map=pos_map_e, e_f=e_f)
            encs.append(x)
            x = down(x)

        x_prep = self.conv_prep(x)
        pos_map_t = F.interpolate(pos_map, scale_factor=1 / 2 ** self.u_depth, mode='bilinear', align_corners=False) if (self.positional_dfe and pos_map is not None) else None
        x_after_body = self.forward_features(x_prep, bokeh_strength=bokeh_strength, pos_map=pos_map_t)
        res = self.conv_after_body(x_after_body) + x_prep

        res = torch.cat((res, pos_map_t), dim=1) if self.positional_conv_last and pos_map_t is not None else res
        x = x + self.conv_last(res)

        for decoder, up, skip, enc_skip, use_pos_map_d, depth_lvl in zip(self.decoders, self.ups, self.skips, encs[::-1], self.dec_blks_use_pos_map, range(0, self.u_depth).__reversed__()):
            pos_map_d = F.interpolate(pos_map, scale_factor=1 / 2 ** depth_lvl, mode='bilinear', align_corners=False) if (use_pos_map_d and pos_map is not None) else None
            x = up(x)
            x = skip(x, enc_skip)
            for blk in decoder:
                x = blk(x, pos_map=pos_map_d, e_f=e_f)

        x = torch.cat((x, pos_map), dim=1) if self.out_stage_use_pos_map and pos_map is not None else x
        x = self.out_stage(x)
        base_out = ((x / self.img_range + self.mean) + source).clamp(0, 1)

        if self.use_refinement and depth is not None and mask is not None:
            F_map, W_focus = self.prior_gen(depth, mask)
            fused = self.fusion_stem(base_out, depth, mask, F_map, W_focus)
            fused_size = (fused.shape[2], fused.shape[3])
            
            pos_map_ref = F.interpolate(pos_map, size=fused_size, mode='bilinear', align_corners=False) if pos_map is not None else None
            
            fused_bhwc = fused.permute(0, 2, 3, 1)
            refined_features_bhwc = self.refinement_blocks(fused_bhwc, fused_size, pos_map=pos_map_ref, att_range_factor=bokeh_strength)
            refined_features = refined_features_bhwc.permute(0, 3, 1, 2)
            refinement_delta = self.refinement_out(refined_features)
            # SCALED RESIDUAL INJECTION: Safe additive convergence
            return base_out + (self.res_scale * refinement_delta * (1 - W_focus))

        return base_out


# ==============================================================================
# 4. DATASET LOADER & CUSTOM LOSSES
# ==============================================================================
class LocalBokehDataset(Dataset):
    def __init__(self, root_dir, split="train", img_size=512):
        self.root      = os.path.join(root_dir, split)
        self.img_size  = img_size
        self.split     = split
        self.to_tensor = transforms.ToTensor()
        self.samples   = []

        meta_dir  = os.path.join(self.root, "metadata")
        rgb_dir   = os.path.join(self.root, "rgb")
        bokeh_dir = os.path.join(self.root, "bokeh")

        if os.path.exists(meta_dir) and len(os.listdir(meta_dir)) > 0:
            json_files = sorted([f for f in os.listdir(meta_dir) if f.endswith('.json')])
            for jf in json_files:
                try:
                    with open(os.path.join(meta_dir, jf), 'r') as f:
                        meta = json.load(f)
                    src_path = os.path.join(self.root, meta['source_image'])
                    for i, tgt_rel in enumerate(meta['target_images']):
                        tgt_path = os.path.join(self.root, tgt_rel)
                        self.samples.append({
                            "source": src_path,
                            "target": tgt_path,
                            "aperture": float(meta['target_avs'][i])
                        })
                except Exception:
                    pass
        elif os.path.exists(rgb_dir) and os.path.exists(bokeh_dir):
            for rgb_name in sorted(os.listdir(rgb_dir)):
                if not rgb_name.endswith('.png'): continue
                group_id = os.path.splitext(rgb_name)[0]
                src_path = os.path.join(rgb_dir, rgb_name)

                bokeh_matches = glob.glob(os.path.join(bokeh_dir, f"{group_id}_f*.png"))
                for b_path in bokeh_matches:
                    m = re.search(r'_f([0-9.]+)\.png', os.path.basename(b_path))
                    ap = float(m.group(1)) if m else 2.8
                    self.samples.append({
                        "source": src_path,
                        "target": b_path,
                        "aperture": ap
                    })
        else:
            pngs = glob.glob(os.path.join(self.root, "**", "*.png"), recursive=True)
            for p in pngs:
                if "target" in p or "bokeh" in p: continue
                self.samples.append({"source": p, "target": p, "aperture": 2.8})

        print(f"[{split}] {len(self.samples)} sample pairs loaded from {self.root}.")

    def get_pos_map(self, w, h, i=0, j=0, crop_size=None):
        if crop_size is not None:
            x_lin = torch.linspace(j / max(1, w-1), (j + crop_size - 1) / max(1, w-1), crop_size)
            y_lin = torch.linspace(1 - i / max(1, h-1), 1 - (i + crop_size - 1) / max(1, h-1), crop_size)
            return torch.meshgrid(x_lin, y_lin, indexing='xy')
        if w > h:
            cd = (1 - h/w) / 2
            return torch.meshgrid(torch.linspace(0,1,w), torch.linspace(1-cd,cd,h), indexing='xy')
        return torch.meshgrid(torch.linspace(0,1,w), torch.linspace(1,0,h), indexing='xy')

    def _load_depth(self, path, h, w):
        raw = cv2.imread(path, cv2.IMREAD_UNCHANGED)
        if raw is not None and raw.dtype == np.uint16:
            return raw.astype(np.float32) / 65535.0
        if os.path.exists(path):
            return np.array(Image.open(path).convert('L'), dtype=np.float32) / 255.0
        return np.full((h, w), 0.5, dtype=np.float32)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s        = self.samples[idx]
        sharp    = Image.open(s['source']).convert('RGB')
        bokeh    = Image.open(s['target']).convert('RGB')
        w, h     = sharp.size
        aperture = s['aperture']

        base_name  = os.path.splitext(os.path.basename(s['source']))[0]
        depth_path = os.path.join(self.root, "depth", base_name + ".png")
        mask_path  = os.path.join(self.root, "mask",  base_name + ".png")

        depth_np = self._load_depth(depth_path, h, w)
        mask_img = Image.open(mask_path).convert('L') if os.path.exists(mask_path) else Image.fromarray(np.ones((h, w), dtype=np.uint8) * 255)

        ap_embed = 2.0 / max(aperture, 1e-4)
        focal_d  = float(np.median(depth_np))
        coc_np   = np.clip(np.abs(depth_np - focal_d) / max(aperture, 1e-4), 0.0, 1.0)

        sharp_t   = self.to_tensor(sharp)
        bokeh_t   = self.to_tensor(bokeh)
        depth_t   = torch.from_numpy(depth_np).unsqueeze(0).float()
        mask_t    = self.to_tensor(mask_img)
        coc_t     = torch.from_numpy(coc_np).unsqueeze(0).float()
        bokeh_map = torch.full((1, h, w), ap_embed, dtype=torch.float32)

        sz = self.img_size
        if self.split == "train":
            ci, cj, ch, cw = transforms.RandomCrop.get_params(sharp_t, (sz, sz))
            def crop(t): return TF.crop(t, ci, cj, ch, cw)
            sharp_t, bokeh_t, bokeh_map, depth_t, mask_t, coc_t = map(crop, [sharp_t, bokeh_t, bokeh_map, depth_t, mask_t, coc_t])
            px, py = self.get_pos_map(w, h, ci, cj, sz)
            pos_map = torch.cat([px.unsqueeze(0), py.unsqueeze(0)], 0)
            if random.random() > 0.5:
                sharp_t, bokeh_t, pos_map, bokeh_map, depth_t, mask_t, coc_t = [TF.hflip(t) for t in [sharp_t, bokeh_t, pos_map, bokeh_map, depth_t, mask_t, coc_t]]
            if random.random() > 0.5:
                sharp_t, bokeh_t, pos_map, bokeh_map, depth_t, mask_t, coc_t = [TF.vflip(t) for t in [sharp_t, bokeh_t, pos_map, bokeh_map, depth_t, mask_t, coc_t]]
        else:
            def ccrop(t): return TF.center_crop(t, (sz, sz))
            sharp_t, bokeh_t, bokeh_map, depth_t, mask_t, coc_t = map(ccrop, [sharp_t, bokeh_t, bokeh_map, depth_t, mask_t, coc_t])
            mi, mj = (h - sz) // 2, (w - sz) // 2
            px, py = self.get_pos_map(w, h, mi, mj, sz)
            pos_map = torch.cat([px.unsqueeze(0), py.unsqueeze(0)], 0)

        return {
            "input": sharp_t,
            "target": bokeh_t,
            "aperture": torch.tensor(ap_embed, dtype=torch.float32),
            "pos_map": pos_map,
            "bokeh_strength_map": bokeh_map,
            "depth": depth_t,
            "mask": mask_t,
            "coc_map": coc_t,
        }

class CharbonnierLoss(nn.Module):
    def __init__(self, eps=1e-3):
        super().__init__()
        self.eps2 = eps ** 2
    def forward(self, pred, target):
        return torch.mean(torch.sqrt((pred - target) ** 2 + self.eps2))

class FFTLoss(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, pred, target):
        pred_fft = torch.fft.rfft2(pred, norm='ortho')
        target_fft = torch.fft.rfft2(target, norm='ortho')
        return torch.mean(torch.abs(pred_fft - target_fft))


# ==============================================================================
# 5. TRAINING SETUP & FULL 4-STAGE ABLATION LOOP (EXACTLY 10 EPOCHS)
# ==============================================================================
IMG_SIZE    = 512
BATCH_SIZE  = 1
ACCUM_STEPS = 2
EPOCHS      = 10          # EXACTLY 10 EPOCHS PER EXPERIMENT
LR_BACKBONE = 1e-5
LR_HAFT     = 5e-4
WARMUP_EPS  = 1           # Aggressive 1-epoch warmup for fast 10-epoch convergence

def find_data_root():
    for search_dir in ["/kaggle/input", "dataset", "."]:
        if not os.path.exists(search_dir): continue
        for root, dirs, files in os.walk(search_dir):
            if "train" in dirs:
                train_path = os.path.join(root, "train")
                if os.path.exists(os.path.join(train_path, "metadata")) or os.path.exists(os.path.join(train_path, "rgb")):
                    return root
    return "dataset/RealBokeh_3MP"

DATA_ROOT = find_data_root()
print(f"Auto-located DATA_ROOT: {DATA_ROOT}")

# --- COMPLETE ABLATION MATRIX (All 4 Core Stages) ---
EXPERIMENTS = [
    ("4_No_Positional_Map",  True,  True,  False),
]

char_fn  = CharbonnierLoss(eps=1e-3).to(device)
fft_fn   = FFTLoss().to(device)
lpips_fn = lpips_lib.LPIPS(net='alex').to(device).eval()
ssim_fn  = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)

train_ds     = LocalBokehDataset(DATA_ROOT, split="train", img_size=IMG_SIZE)
val_ds       = LocalBokehDataset(DATA_ROOT, split="test",  img_size=IMG_SIZE)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=1, shuffle=False, num_workers=2, pin_memory=True)

def calc_psnr(pred, target):
    mse = torch.mean((pred.float() - target.float()) ** 2)
    return 20 * math.log10(1.0 / (mse.item() ** 0.5 + 1e-8))

os.makedirs("kaggle_checkpoints", exist_ok=True)
results_summary = {}

for exp_name, refine_flag, coc_flag, pos_flag in EXPERIMENTS:
    print("\n" + "="*70)
    print(f"STARTING EXPERIMENT: {exp_name} ({EPOCHS} EPOCHS)")
    print(f"Config -> Refinement: {refine_flag} | CoC Map: {coc_flag} | Positional Map: {pos_flag}")
    print("="*70)
    
    MODEL_ARGS = dict(
        u_width=32, u_depth=2,
        in_stage_use_pos_map=pos_flag,
        in_stage_use_bokeh_strength_map=True,
        enc_blks_use_pos_map=[pos_flag, pos_flag],
        dec_blks_use_pos_map=[pos_flag, pos_flag],
        out_stage_use_pos_map=pos_flag,
        use_refinement=refine_flag,
        use_coc_map=coc_flag,
        positional_dfe=pos_flag,
        positional_conv_last=pos_flag,
        embed_dims=[192]*6, depths=[6]*6, num_heads=[6]*6, mlp_ratios=[2]*6,
        init_values=[2]*6, heads_ranges=[9]*9, dec_blk_nums=[2, 4],
        in_chans=3, use_checkpoints=[True]*6,
    )

    model = HAFT_Bokehlicious(**MODEL_ARGS).to(device)

    # Robust state dict loading
    large_pt = glob.glob("/kaggle/input/**/large.pt", recursive=True) + glob.glob("checkpoints/large.pt")
    if large_pt:
        ckpt_path = large_pt[0]
        state = torch.load(ckpt_path, map_location=device, weights_only=False)
        mdict = model.state_dict()
        matched = {}
        for k, v in state.items():
            if k not in mdict: continue
            if v.shape == mdict[k].shape:
                matched[k] = v
            elif k == 'in_stage.weight':
                new_w = mdict[k].clone()
                min_ch = min(v.shape[1], mdict[k].shape[1])
                new_w[:, :min_ch, :, :] = v[:, :min_ch, :, :]
                matched[k] = new_w
        model.load_state_dict(matched, strict=False)

    haft_keys       = {'aperture_encoder', 'prior_gen', 'fusion_stem', 'refinement'}
    backbone_params = [p for n,p in model.named_parameters() if p.requires_grad and not any(h in n for h in haft_keys)]
    haft_params     = [p for n,p in model.named_parameters() if p.requires_grad and any(h in n for h in haft_keys)]

    optimizer = optim.AdamW([
        {'params': backbone_params, 'lr': LR_BACKBONE, 'initial_lr': LR_BACKBONE},
        {'params': haft_params,     'lr': LR_HAFT,     'initial_lr': LR_HAFT},
    ], weight_decay=1e-4)

    # FAST WARMUP + COSINE SCHEDULER: Tuned specifically for 10-epoch runs
    def lr_lambda(current_epoch):
        if current_epoch < WARMUP_EPS:
            return float(current_epoch + 1) / float(max(1, WARMUP_EPS))
        progress = float(current_epoch - WARMUP_EPS) / float(max(1, EPOCHS - WARMUP_EPS))
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

    best_psnr  = 0.0
    best_ssim  = 0.0
    best_lpips = float('inf')

    gc.collect()
    torch.cuda.empty_cache()

    try:
        for epoch in range(1, EPOCHS + 1):
            model.train()
            optimizer.zero_grad()
            total_loss = 0.0
            pbar = tqdm(train_loader, desc=f"[{exp_name}] Ep {epoch}/{EPOCHS}", leave=False)

            for i, batch in enumerate(pbar):
                inp   = batch["input"].to(device, non_blocking=True)
                tgt   = batch["target"].to(device, non_blocking=True)
                ap    = batch["aperture"].to(device, non_blocking=True)
                pos   = batch["pos_map"].to(device, non_blocking=True)
                b_map = batch["bokeh_strength_map"].to(device, non_blocking=True)
                depth = batch["depth"].to(device, non_blocking=True)
                mask  = batch["mask"].to(device, non_blocking=True)
                coc   = batch["coc_map"].to(device, non_blocking=True)

                with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                    pred = model(inp, bokeh_strength=ap, pos_map=pos, bokeh_strength_map=b_map, depth=depth, mask=mask, coc_map=coc)
                    loss_c = char_fn(pred, tgt)

                loss_extra = torch.tensor(0.0, device=device)
                if (i + 1) % ACCUM_STEPS == 0:
                    p32 = pred.float().clamp(0, 1)
                    t32 = tgt.float()
                    loss_extra = 0.1 * fft_fn(p32, t32) + 0.3 * lpips_fn(p32*2-1, t32*2-1).mean()

                loss = (loss_c + loss_extra) / ACCUM_STEPS
                if torch.isnan(loss) or torch.isinf(loss):
                    optimizer.zero_grad()
                    continue

                loss.backward()

                if (i + 1) % ACCUM_STEPS == 0:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                    optimizer.zero_grad()

                total_loss += loss.item() * ACCUM_STEPS
                pbar.set_postfix({"L": f"{loss.item()*ACCUM_STEPS:.4f}"})

            scheduler.step()
            avg_train = total_loss / len(train_loader)

            model.eval()
            val_psnr, val_ssim_sum, val_lpips_sum = 0.0, 0.0, 0.0
            with torch.no_grad():
                for batch in val_loader:
                    inp   = batch["input"].to(device)
                    tgt   = batch["target"].to(device)
                    ap    = batch["aperture"].to(device)
                    pos   = batch["pos_map"].to(device)
                    b_map = batch["bokeh_strength_map"].to(device)
                    depth = batch["depth"].to(device)
                    mask  = batch["mask"].to(device)
                    coc   = batch["coc_map"].to(device)

                    with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                        pred = model(inp, bokeh_strength=ap, pos_map=pos, bokeh_strength_map=b_map, depth=depth, mask=mask, coc_map=coc)
                    pred = pred.float().clamp(0, 1)
                    val_psnr += calc_psnr(pred, tgt)
                    val_ssim_sum += ssim_fn(pred, tgt.float()).item()
                    val_lpips_sum += lpips_fn(pred*2-1, tgt.float()*2-1).mean().item()

            avg_psnr  = val_psnr  / max(len(val_loader), 1)
            avg_ssim  = val_ssim_sum / max(len(val_loader), 1)
            avg_lpips = val_lpips_sum / max(len(val_loader), 1)

            if avg_psnr > best_psnr:
                best_psnr = avg_psnr
                torch.save(model.state_dict(), f"kaggle_checkpoints/{exp_name}_best_psnr.pth")

            if avg_ssim > best_ssim:
                best_ssim = avg_ssim

            if avg_lpips < best_lpips:
                best_lpips = avg_lpips
                torch.save(model.state_dict(), f"kaggle_checkpoints/{exp_name}_best_lpips.pth")

            print(f"[{exp_name}] Ep {epoch:2d}/{EPOCHS} | Train Loss {avg_train:.4f} | Val PSNR {avg_psnr:.2f} dB (Best: {best_psnr:.2f}) | Val SSIM {avg_ssim:.4f} (Best: {best_ssim:.4f}) | Val LPIPS {avg_lpips:.4f} (Best: {best_lpips:.4f})")

            gc.collect()
            torch.cuda.empty_cache()

    except KeyboardInterrupt:
        print(f"\n[WARNING] {exp_name} interrupted manually!")
        torch.save(model.state_dict(), f"kaggle_checkpoints/{exp_name}_interrupted.pth")
        break

    results_summary[exp_name] = {
        "Best_PSNR_dB": round(best_psnr, 2), 
        "Best_SSIM": round(best_ssim, 4), 
        "Best_LPIPS": round(best_lpips, 4)
    }

    del model, optimizer, scheduler
    gc.collect()
    torch.cuda.empty_cache()

print("\n" + "="*80)
print("ALL 4 ABLATION EXPERIMENTS COMPLETED!")
print("="*80)
for exp, metrics in results_summary.items():
    print(f"{exp:<25} | Best PSNR: {metrics['Best_PSNR_dB']:>5} dB | Best SSIM: {metrics['Best_SSIM']:>6} | Best LPIPS: {metrics['Best_LPIPS']:>6}")


# ==============================================================================
# 6. FINAL EVALUATION — PSNR / SSIM / LPIPS
# ==============================================================================
print("\nLoading best checkpoint for final evaluation...")
model = HAFT_Bokehlicious(
    u_width=32, u_depth=2,
    in_stage_use_pos_map=True,
    in_stage_use_bokeh_strength_map=True,
    enc_blks_use_pos_map=[True, True],
    dec_blks_use_pos_map=[True, True],
    out_stage_use_pos_map=True,
    use_refinement=True,
    use_coc_map=True,
    positional_dfe=True,
    positional_conv_last=True,
    embed_dims=[192]*6, depths=[6]*6, num_heads=[6]*6, mlp_ratios=[2]*6,
    init_values=[2]*6, heads_ranges=[9]*9, dec_blk_nums=[2, 4],
    in_chans=3, use_checkpoints=[True]*6,
).to(device)

model.load_state_dict(
    torch.load("kaggle_checkpoints/1_Full_HAFT_Baseline_best_psnr.pth", map_location=device) if os.path.exists("kaggle_checkpoints/1_Full_HAFT_Baseline_best_psnr.pth") else print("Checkpoint not found, continuing...")
)
model.eval()

psnr_m  = PeakSignalNoiseRatio(data_range=1.0).to(device)
ssim_m  = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)
lpips_m = lpips_lib.LPIPS(net='alex').to(device).eval()

test_ds     = LocalBokehDataset(DATA_ROOT, split="test", img_size=IMG_SIZE)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=2)

tp, ts, tl = 0.0, 0.0, 0.0
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Final Eval"):
        inp   = batch["input"].to(device)
        tgt   = batch["target"].to(device)
        ap    = batch["aperture"].to(device)
        pos   = batch["pos_map"].to(device)
        b_map = batch["bokeh_strength_map"].to(device)
        depth = batch["depth"].to(device)
        mask  = batch["mask"].to(device)
        coc   = batch["coc_map"].to(device)

        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            pred = model(inp, bokeh_strength=ap, pos_map=pos,
                         bokeh_strength_map=b_map, depth=depth,
                         mask=mask, coc_map=coc)
            
        pred = pred.float().clamp(0, 1)
        tgt  = tgt.float()

        tp += psnr_m(pred, tgt).item()
        ts += ssim_m(pred, tgt).item()
        tl += lpips_m(pred * 2 - 1, tgt * 2 - 1).mean().item()

n = max(len(test_loader), 1)
final_psnr  = tp / n
final_ssim  = ts / n
final_lpips = tl / n

results = f"""# HAFT Final Results — {EPOCHS} Epochs @ {IMG_SIZE}×{IMG_SIZE}

| Metric | Score |
|---|---|
| **PSNR**  | {final_psnr:.2f} dB |
| **SSIM**  | {final_ssim:.4f}    |
| **LPIPS** | {final_lpips:.4f}   |
"""

print(results)
with open("kaggle_final_results.md", "w") as f:
    f.write(results)
print("Saved kaggle_final_results.md successfully.")